# Experiment 3: Circuit Discovery & Span Validation

**Criterion 3**: Within-span similarity elevation > population mean + 1σ
**Criterion 4**: Circuit diversity ≥ 60% layer coverage
**Criterion 5**: Class purity distribution is bimodal

Runs the full span-centric circuit discovery pipeline, validates discovered circuits,
and checks multi-circuit membership.

## Colab Setup
Clones the repo, installs dependencies, and mounts Google Drive.

In [ ]:
import os, sys

# 1. Clone repository
REPO_DIR = '/content/trainable-circuits'
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/jacobposchl/trainable-circuits {REPO_DIR}

os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)

# 2. Install extra dependencies
!pip install hdbscan umap-learn -q

# 3. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# 4. Path constants
DRIVE_DIR  = '/content/drive/MyDrive/ctls'
DATA_DIR   = DRIVE_DIR + '/data/raw'
CKPT_DIR   = DRIVE_DIR + '/experiments'
CONFIG_DIR = REPO_DIR  + '/configs'

print('Repo:     ', REPO_DIR)
print('Data:     ', DATA_DIR)
print('Checkpts: ', CKPT_DIR)

In [ ]:
import torch
import yaml
import numpy as np
import matplotlib.pyplot as plt
from models.backbone import FrozenBackbone
from models.meta_encoder import MetaEncoder
from data.cifar import get_standard_loaders
from evaluation.circuit_analysis import CircuitAnalyzer
from evaluation.discovery import SpanCentricDiscovery
from evaluation.metrics import (
    within_span_elevation,
    circuit_diversity,
    class_purity_distribution,
)
from evaluation.circuit_viz import (
    plot_span_coverage,
    plot_multi_circuit_histogram,
    plot_circuit_members,
)
from evaluation.circuit_analysis import denormalize

In [ ]:
# Load config and override data directory
config_path     = CONFIG_DIR + '/phase1.yaml'
checkpoint_path = CKPT_DIR  + '/phase1/best.pt'

with open(config_path) as f:
    config = yaml.safe_load(f)

config['data']['data_dir'] = DATA_DIR

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

In [ ]:
# Build models and load weights
backbone = FrozenBackbone(
    arch=config['model']['arch'],
    num_classes=config['model']['num_classes'],
    pretrained=config['model']['pretrained'],
).to(device)

meta_encoder = MetaEncoder(
    layer_dims=backbone.layer_dims,
    projection_dim=config['model']['meta_encoder']['projection_dim'],
    n_heads=config['model']['meta_encoder']['n_heads'],
    n_transformer_layers=config['model']['meta_encoder']['n_transformer_layers'],
    dropout=config['model']['meta_encoder']['dropout']
).to(device)

ckpt = torch.load(checkpoint_path, map_location=device, weights_only=False)
meta_encoder.load_state_dict(ckpt['meta_encoder_state'])
meta_encoder.eval()
print('Model loaded.')

In [ ]:
# Create validation loader, collect representations, compute pair profiles
_, val_loader = get_standard_loaders(
    data_dir=DATA_DIR,
    batch_size=config['data'].get('batch_size', 256),
    num_workers=0,
    augment=False,
    download=False,
)

analyzer = CircuitAnalyzer(backbone, meta_encoder, val_loader, device)
data     = analyzer.collect_representations(max_samples=2000)
print(f'Collected {data["labels"].shape[0]} samples')

# Pair indices (subsample to 50k)
N = data['z_list'][0].shape[0]
idx_a, idx_b = torch.triu_indices(N, N, offset=1)
MAX_PAIRS = 50_000
if idx_a.shape[0] > MAX_PAIRS:
    perm  = torch.randperm(idx_a.shape[0])[:MAX_PAIRS]
    idx_a, idx_b = idx_a[perm], idx_b[perm]

# True pair profiles from flow targets [N_pairs, L]
true_sims    = CircuitAnalyzer.compute_pair_profiles(data['flow_targets'], idx_a, idx_b)
true_sims_np = true_sims.numpy()
pair_indices = np.stack([idx_a.numpy(), idx_b.numpy()], axis=1)  # [N_pairs, 2]
L            = len(data['z_list'])

print(f'N pairs: {idx_a.shape[0]}, L layers: {L}')

In [ ]:
# Run span-centric circuit discovery
discovery = SpanCentricDiscovery(
    n_layers=L,
    min_cluster_fraction=config['discovery']['min_cluster_fraction'],
    max_cluster_fraction=config['discovery'].get('max_cluster_fraction', 0.40),
    min_cluster_size=config['discovery']['hdbscan_min_cluster_size'],
)

circuits = discovery.discover_all(true_sims_np, pair_indices)
print(f'Discovered {len(circuits)} canonical circuits')

In [ ]:
# Criterion 3: Within-span similarity elevation
# within_span_elevation(cluster_similarities, population_similarities)
# both are 1-D arrays of mean within-span similarity per pair
for i, circuit in enumerate(circuits):
    l_start, l_end = circuit['span']
    span_sims_all     = true_sims_np[:, l_start:l_end+1].mean(axis=1)  # [N_pairs]
    cluster_mask      = circuit['pair_mask']
    span_sims_cluster = span_sims_all[cluster_mask]

    result = within_span_elevation(span_sims_cluster, span_sims_all)
    status = '\u2713' if result['passes'] else '\u2717'
    print(f"Circuit {i} span {circuit['span']}: "
          f"elevation = {result['elevation_sigma']:.2f}\u03c3  {status}")

In [ ]:
# Criterion 4: Circuit diversity
# circuit_diversity expects list of (l_start, l_end) spans
spans  = [c['span'] for c in circuits]
result = circuit_diversity(spans, L)
print(f"Layer coverage: {result['coverage']:.1%}  "
      f"(target \u2265 60%, {'PASS' if result['passes'] else 'FAIL'})")

fig = plot_span_coverage(circuits, L)
plt.tight_layout()
plt.show()

In [ ]:
# Criterion 5: Class purity distribution
# class_purity_distribution expects list[float] of purity scores
labels_np = data['labels'].numpy()
purities  = [
    SpanCentricDiscovery.compute_class_purity(c, pair_indices, labels_np)
    for c in circuits
]

result = class_purity_distribution(purities)
print(f"Agnostic (<0.3): {result['n_agnostic']}, "
      f"Specific (>0.7): {result['n_specific']}  "
      f"({'PASS' if result['passes'] else 'FAIL'})")

In [ ]:
# Multi-circuit membership distribution
n_pairs = true_sims_np.shape[0]
counts  = SpanCentricDiscovery.multi_circuit_membership(circuits, n_pairs)

fig = plot_multi_circuit_histogram(counts)
plt.tight_layout()
plt.show()

In [ ]:
# Visualize example circuits (first 3)
# plot_circuit_members(images, labels, per_input_profiles, span)
# Build per-input mean profile from circuit pairs
for i, circuit in enumerate(circuits[:3]):
    mask         = circuit['pair_mask']
    circuit_pairs = pair_indices[mask]            # [N_cluster, 2]
    unique_inputs = np.unique(circuit_pairs)      # input indices
    n_show        = min(16, len(unique_inputs))
    show_idx      = unique_inputs[:n_show]

    # Per-input mean profile: average true_sims over pairs this input belongs to
    input_profiles = np.zeros((n_show, L))
    for j, inp in enumerate(show_idx):
        involved = np.where((circuit_pairs[:, 0] == inp) | (circuit_pairs[:, 1] == inp))[0]
        if len(involved) > 0:
            input_profiles[j] = true_sims_np[mask][involved].mean(axis=0)

    imgs_np = denormalize(data['images'][show_idx]).numpy()  # [n_show, 3, 32, 32]
    lbls_np = data['labels'][show_idx].numpy()

    fig = plot_circuit_members(imgs_np, lbls_np, input_profiles, span=circuit['span'])
    plt.suptitle(f"Circuit {i}  (span {circuit['span']})", y=1.01)
    plt.tight_layout()
    plt.show()